# Notebook 1: Simple Movie Chatbot

## Learning Objectives
- Understand basic chatbot architecture
- Implement conversation memory
- Use model-agnostic LLM loading
- Load configuration from Excel files

## What's New in This Notebook
- Basic conversational AI
- System prompts from Excel configuration
- Memory-enabled conversations
- Works with any LLM provider (OpenAI, Gemini, Mistral, Claude)

## Prerequisites
- Python 3.11+
- API key for your chosen LLM provider
- Use case Excel file

## Step 1: Install Required Packages

In [ ]:
# Install all required packages
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai --quiet

print("✓ Packages installed successfully")

## Step 2: Import Libraries

In [1]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

# Import our utilities
from ai_course_utils import (
    load_llm_from_env,
    load_use_case_config,
    display_config
)

print("✓ All imports successful")

✓ All imports successful


## Step 3: Display Configuration

This shows your API configuration from the .env file.

In [2]:
# Display current configuration
display_config()

API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')


## Step 4: Load Use Case Configuration

**USER INPUT REQUIRED**: Provide the path to your use case Excel file.

In [3]:
# ============================================================================
# USER INPUT: Specify the path to your use case file
# ============================================================================
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"  # ← CHANGE THIS PATH

# Load configuration
use_case_config = load_use_case_config(use_case_file)

# Extract system prompt
system_prompt = use_case_config.get("agent_prompt", "You are a helpful movie assistant")

print(f"\n📋 System Prompt Preview:")
print(system_prompt[:200] + "...")
print(f"\n✓ Loaded from: {use_case_file}")

✓ Use case loaded: ../data/use_case_Movie_Recommendation.xlsx
  Components: user, agent_prompt, url

📋 System Prompt Preview:
You are a movie expert whose primary goal is to help users with searching for movies. You are friendly and concise. You only provide factual answers to queries, and do not provide answers that are not...

✓ Loaded from: ../data/use_case_Movie_Recommendation.xlsx


## Step 5: Initialize LLM

The LLM is loaded based on your .env configuration. This works with any provider!

In [5]:
import os
from pathlib import Path

# Check for .env file
print("Current directory:", os.getcwd())
print("\nLooking for .env file...")

# Check current directory
if os.path.exists('.env'):
    print("✓ .env found in current directory")
elif os.path.exists('../.env'):
    print("✓ .env found in parent directory")
else:
    print("❌ .env file NOT FOUND")
    print("\nSearching nearby...")
    for root, dirs, files in os.walk('..'):
        if '.env' in files:
            print(f"  Found: {os.path.join(root, '.env')}")

Current directory: /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/ai-powered-move-recommendation/notebooks

Looking for .env file...
✓ .env found in parent directory


In [6]:
# Load LLM from environment configuration
llm = load_llm_from_env()
print("✓ LLM initialized and ready")

🤖 Loading LLM: openai / gpt-4.1-mini
✓ LLM initialized and ready


## Step 6: Build Chatbot Graph

In [7]:
def chatbot(state: MessagesState) -> dict:
    """
    Simple chatbot that maintains conversation history.
    
    Args:
        state: Current conversation state with message history
        
    Returns:
        Updated state with new message
    """
    # Prepend system message to conversation
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    
    # Get response from LLM
    response = llm.invoke(messages)
    
    return {"messages": [response]}

# Build the graph
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

# Compile with memory to maintain conversation history
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("✓ Chatbot graph built successfully")
print("✓ Memory enabled for conversation history")

✓ Chatbot graph built successfully
✓ Memory enabled for conversation history


## Step 7: Create Helper Function

In [8]:
def chat(user_input: str, thread_id: str = "default") -> str:
    """
    Send a message to the chatbot and get a response.
    
    Args:
        user_input: The user's message
        thread_id: Conversation thread ID (use same ID to maintain context)
        
    Returns:
        The chatbot's response as a string
    """
    config = {"configurable": {"thread_id": thread_id}}
    
    result = graph.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config
    )
    
    return result["messages"][-1].content

print("🎬 Movie Chatbot Ready!")
print("\nUsage: response = chat('Your message here')")

🎬 Movie Chatbot Ready!

Usage: response = chat('Your message here')


## Step 8: Test the Chatbot

Let's test with some example queries.

In [9]:
# Example 1: Simple recommendation
print("=" * 70)
print("Example 1: Simple Recommendation")
print("=" * 70)
print("\nUser: Recommend a sci-fi movie")
response1 = chat("Recommend a sci-fi movie", thread_id="example1")
print(f"Bot: {response1}")

Example 1: Simple Recommendation

User: Recommend a sci-fi movie
Bot: Great choice! Sci-fi is a broad genre. Are you looking for something more action-packed, thought-provoking, or maybe a classic? Also, do you prefer recent movies or older ones?


In [10]:
# Example 2: Follow-up question (tests memory)
print("\n" + "=" * 70)
print("Example 2: Follow-up Question (Testing Memory)")
print("=" * 70)
print("\nUser: What about something with time travel?")
response2 = chat("What about something with time travel?", thread_id="example1")
print(f"Bot: {response2}")


Example 2: Follow-up Question (Testing Memory)

User: What about something with time travel?
Bot: Time travel is a fascinating theme! Do you prefer a serious, mind-bending story like "Primer," or something more adventurous and fun like "Back to the Future"? Also, are you interested in a movie with a strong romantic subplot or more focused on the sci-fi elements?


In [11]:
# Example 3: New conversation thread
print("\n" + "=" * 70)
print("Example 3: New Conversation")
print("=" * 70)
print("\nUser: What's a good family movie?")
response3 = chat("What's a good family movie?", thread_id="example3")
print(f"Bot: {response3}")


Example 3: New Conversation

User: What's a good family movie?
Bot: Great! To help me suggest the best family movie for you, could you tell me a bit more? Are you looking for something animated or live-action? Also, do you have a preferred decade or any favorite genres like comedy, adventure, or fantasy?


## Step 9: Interactive Testing

Try your own queries here!

In [12]:
# YOUR TURN: Test the chatbot
# Modify the query below and run this cell

my_query = "What are some classic action movies?"
print(f"User: {my_query}")
print(f"Bot: {chat(my_query, thread_id='my_session')}")

User: What are some classic action movies?
Bot: Classic action movies often feature thrilling sequences, iconic heroes, and memorable villains. Are you interested in classics from a particular decade, like the '80s or '90s? Or do you prefer action movies from a specific country or with a certain style, like martial arts or spy thrillers? This will help me tailor the list for you!


## Summary

### What We Built:
✅ Simple conversational chatbot  
✅ Model-agnostic (works with any LLM)  
✅ Configuration from Excel file  
✅ Memory-enabled conversations  
✅ Clean, reusable code  

### Key Concepts:
- **System Prompt**: Defines chatbot personality and behavior
- **Conversation Memory**: Maintains context across messages
- **Thread ID**: Separate conversations with different IDs
- **Model-Agnostic**: Switch LLM providers via .env file

### Next Steps:
In **Notebook 2**, we'll add web search capability to access current information!

### Experiments to Try:
1. Change the system prompt in your Excel file
2. Test with different LLM providers (change .env)
3. Try multiple conversation threads
4. Test memory by asking follow-up questions